# 🤖 QuantumFinance — AI Agent para Recomendação de Investimentos

**Projeto Acadêmico — FIAP Pós Tech AI for Devs**

Neste notebook vamos construir um agente de IA capaz de:
- Coletar dados de ações da B3 (VALE3, PETR4, BBAS3, ITUB4)
- Analisar notícias financeiras em tempo real
- Calcular indicadores técnicos (RSI, MACD, Bollinger Bands, etc.)
- Gerar recomendações de COMPRAR / VENDER / AGUARDAR com justificativa

---
**Arquitetura:** Multi-Agent com padrão ReAct (Reasoning + Acting)

## 📦 Parte 1 — Instalação das Dependências

Primeiro precisamos instalar todas as bibliotecas que vamos usar.
Se já instalou via `pip install -r requirements.txt`, pode pular esta célula.

In [ ]:
# Instala todas as dependências necessárias
# O ! no início significa que é um comando do terminal rodando dentro do notebook

!pip install yfinance feedparser pandas numpy vaderSentiment plotly langchain langchain-openai langgraph python-dotenv requests -q

print('✅ Dependências instaladas com sucesso!')

## 📚 Parte 2 — Importações

Aqui importamos tudo que vamos precisar ao longo do notebook.

In [ ]:
# Bibliotecas padrão do Python
import os
import sys
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')  # Ignora warnings pra deixar mais limpo

# Manipulação de dados
import pandas as pd
import numpy as np

# Dados de mercado financeiro
import yfinance as yf

# Notícias via RSS
import feedparser

# Análise de sentimento
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Visualizações interativas
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Adiciona o diretório raiz ao path pra importar nossos módulos
sys.path.insert(0, '..')

print('✅ Importações realizadas!')
print(f'   pandas: {pd.__version__}')
print(f'   numpy: {np.__version__}')
print(f'   yfinance: {yf.__version__}')

## 📈 Parte 3 — Coleta de Dados de Mercado

Vamos baixar os dados históricos das 4 ações monitoradas usando o `yfinance`.
O Yahoo Finance disponibiliza dados de graça — perfeito pra projetos acadêmicos!

In [ ]:
# Tickers que vamos analisar
# No Yahoo Finance, ações brasileiras precisam do sufixo '.SA'
TICKERS = {
    'VALE3': 'VALE3.SA',
    'PETR4': 'PETR4.SA',
    'BBAS3': 'BBAS3.SA',
    'ITUB4': 'ITUB4.SA',
}

# Vamos baixar 6 meses de dados (suficiente pra calcular todos os indicadores)
PERIODO = '6mo'

# Dicionário pra guardar os dados de cada ação
dados_mercado = {}

for nome, ticker_yf in TICKERS.items():
    print(f'📥 Baixando dados de {nome}...')
    
    # yf.Ticker cria um objeto com todos os dados do ativo
    ativo = yf.Ticker(ticker_yf)
    
    # .history() retorna um DataFrame com OHLCV
    # (Open, High, Low, Close, Volume)
    hist = ativo.history(period=PERIODO)
    
    if not hist.empty:
        dados_mercado[nome] = hist
        preco_atual = hist['Close'].iloc[-1]
        variacao = ((hist['Close'].iloc[-1] - hist['Close'].iloc[-2]) / hist['Close'].iloc[-2]) * 100
        print(f'   ✅ {len(hist)} dias | Preço: R$ {preco_atual:.2f} | Variação: {variacao:+.2f}%')
    else:
        print(f'   ❌ Erro ao baixar dados de {nome}')

print(f'\n🎯 Dados coletados para {len(dados_mercado)} ações!')

In [ ]:
# Vamos ver como ficam os dados da VALE3
# O DataFrame tem as colunas: Open, High, Low, Close, Volume
print('Primeiros 5 registros da VALE3:')
dados_mercado['VALE3'].head()

In [ ]:
# Estatísticas básicas dos preços
print('Estatísticas dos preços de fechamento:')
resumo = pd.DataFrame({
    ticker: {
        'Preço Atual': f"R$ {df['Close'].iloc[-1]:.2f}",
        'Máximo 6m': f"R$ {df['Close'].max():.2f}",
        'Mínimo 6m': f"R$ {df['Close'].min():.2f}",
        'Média 6m': f"R$ {df['Close'].mean():.2f}",
        'Vol. Médio': f"{df['Volume'].mean():,.0f}"
    }
    for ticker, df in dados_mercado.items()
})
resumo

## 📊 Parte 4 — Cálculo de Indicadores Técnicos

Agora vamos calcular os indicadores técnicos.
Cada indicador nos dá uma "pista" sobre o comportamento futuro do preço.

In [ ]:
# ============================================================
# FUNÇÃO RSI — Relative Strength Index
# Mede a força da tendência atual
# > 70 = sobrecomprado (possível queda)
# < 30 = sobrevendido (possível subida)
# ============================================================

def calcular_rsi(closes, periodo=14):
    """Calcula o RSI usando médias exponenciais."""
    delta = closes.diff()
    ganhos = delta.where(delta > 0, 0.0)
    perdas = -delta.where(delta < 0, 0.0)
    media_ganhos = ganhos.ewm(com=periodo-1, min_periods=periodo).mean()
    media_perdas = perdas.ewm(com=periodo-1, min_periods=periodo).mean()
    rs = media_ganhos / media_perdas
    return 100 - (100 / (1 + rs))

# Testa o RSI na VALE3
rsi_vale = calcular_rsi(dados_mercado['VALE3']['Close'])
rsi_atual = rsi_vale.iloc[-1]
print(f'RSI atual da VALE3: {rsi_atual:.2f}')

if rsi_atual >= 70:
    print('⚠️  SOBRECOMPRADO — cuidado com entrada agora!')
elif rsi_atual <= 30:
    print('✅ SOBREVENDIDO — possível oportunidade de compra!')
else:
    print('📊 RSI NEUTRO — sem sinal claro')

In [ ]:
# ============================================================
# FUNÇÃO MACD — Moving Average Convergence Divergence
# Identifica mudanças na força e direção da tendência
# ============================================================

def calcular_macd(closes, rapido=12, lento=26, sinal=9):
    """Calcula MACD, Signal e Histograma."""
    ema_rapida = closes.ewm(span=rapido, adjust=False).mean()
    ema_lenta = closes.ewm(span=lento, adjust=False).mean()
    macd = ema_rapida - ema_lenta
    signal = macd.ewm(span=sinal, adjust=False).mean()
    hist = macd - signal
    return macd, signal, hist

# Calcula pra VALE3
macd_vale, signal_vale, hist_vale = calcular_macd(dados_mercado['VALE3']['Close'])

print(f'MACD atual VALE3: {macd_vale.iloc[-1]:.4f}')
print(f'Signal atual VALE3: {signal_vale.iloc[-1]:.4f}')

if macd_vale.iloc[-1] > signal_vale.iloc[-1]:
    print('✅ MACD BULLISH — MACD acima do Signal (sinal de compra)')
else:
    print('⚠️  MACD BEARISH — MACD abaixo do Signal (sinal de venda)')

In [ ]:
# ============================================================
# BANDAS DE BOLLINGER
# Mostram os limites de variação "normal" do preço
# ============================================================

def calcular_bollinger(closes, periodo=20, desvios=2):
    """Calcula as 3 bandas de Bollinger."""
    central = closes.rolling(window=periodo).mean()
    std = closes.rolling(window=periodo).std()
    superior = central + (desvios * std)
    inferior = central - (desvios * std)
    return superior, central, inferior

# Calcula pra todos os tickers
print('Posição do preço nas Bandas de Bollinger:')
for ticker, df in dados_mercado.items():
    sup, cen, inf = calcular_bollinger(df['Close'])
    preco = df['Close'].iloc[-1]
    
    if preco >= sup.iloc[-1]:
        posicao = '🔴 Banda Superior (sobrecomprado)'
    elif preco <= inf.iloc[-1]:
        posicao = '🟢 Banda Inferior (sobrevendido/oportunidade)'
    else:
        posicao = '🟡 Dentro das bandas (neutro)'
    
    print(f'  {ticker}: R$ {preco:.2f} → {posicao}')

In [ ]:
# ============================================================
# Usando o módulo do projeto pra calcular TODOS de uma vez
# ============================================================
from src.indicators.technical import calcular_todos_indicadores

print('Calculando todos os indicadores para VALE3...')
indicadores_vale = calcular_todos_indicadores(dados_mercado['VALE3'])

print('\n📊 INDICADORES ATUAIS — VALE3')
print('='*40)
atual = indicadores_vale['atual']
interp = indicadores_vale['interpretacoes']

for chave, valor in atual.items():
    if chave not in ['volume_atual', 'volume_medio']:
        print(f'  {chave:20}: {valor}')

print('\n📝 INTERPRETAÇÕES')
print('='*40)
for chave, valor in interp.items():
    print(f'  {chave:20}: {valor}')

## 📰 Parte 5 — Análise de Notícias

Agora vamos buscar notícias dos portais financeiros e analisar o sentimento.
Usamos o VADER, que é um analisador de sentimento que funciona offline.

In [ ]:
# ============================================================
# ANÁLISE DE SENTIMENTO COM VADER
# O VADER classifica textos em Positivo, Negativo ou Neutro
# e gera um score de -1 (muito negativo) a +1 (muito positivo)
# ============================================================
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Inicializa o VADER
vader = SentimentIntensityAnalyzer()

# Adiciona palavras financeiras em português ao dicionário
palavras_financeiras = {
    'lucro': 2.5, 'crescimento': 2.0, 'alta': 1.5, 'valorização': 2.0,
    'dividendo': 2.0, 'superou': 2.5, 'recorde': 2.5,
    'queda': -1.5, 'perda': -2.0, 'prejuízo': -2.5, 'crise': -2.5,
    'risco': -1.5, 'inadimplência': -2.0
}
vader.lexicon.update(palavras_financeiras)

# Testa com algumas manchetes fictícias
manchetes_teste = [
    'Vale reporta lucro recorde no trimestre e anuncia dividendos',
    'Petrobras enfrenta queda nas ações após mudança na política de preços',
    'Banco do Brasil mantém resultados estáveis no período',
    'Itaú supera expectativas com crescimento de 15% no crédito',
]

print('Teste de análise de sentimento:')
print('='*60)
for manchete in manchetes_teste:
    scores = vader.polarity_scores(manchete)
    compound = scores['compound']
    label = 'POSITIVO' if compound >= 0.05 else 'NEGATIVO' if compound <= -0.05 else 'NEUTRO'
    emoji = '🟢' if label == 'POSITIVO' else '🔴' if label == 'NEGATIVO' else '🟡'
    print(f'{emoji} [{compound:+.3f}] {manchete[:60]}')

In [ ]:
# Busca notícias reais usando o módulo do projeto
from src.sentiment.analyzer import buscar_noticias, analisar_conjunto_noticias

# Vamos buscar notícias da VALE3
print('🔍 Buscando notícias para VALE3...')
noticias_vale = buscar_noticias('VALE3', max_noticias=10)

print(f'\n📰 Notícias encontradas: {len(noticias_vale)}')
print('-'*50)

for i, noticia in enumerate(noticias_vale[:5], 1):
    print(f'{i}. {noticia["titulo"][:70]}')
    print(f'   Fonte: {noticia["fonte"]} | {noticia["data"][:20]}')
    print()

In [ ]:
# Analisa o sentimento do conjunto de notícias
analise_vale = analisar_conjunto_noticias(noticias_vale)

print('📊 ANÁLISE DE SENTIMENTO — VALE3')
print('='*40)
print(f'  Score médio: {analise_vale["score_medio"]:+.3f}')
print(f'  Classificação: {analise_vale["label_geral"]}')
print(f'  Total notícias: {analise_vale["total_noticias"]}')
print(f'  ✅ Positivas: {analise_vale["noticias_positivas"]}')
print(f'  ❌ Negativas: {analise_vale["noticias_negativas"]}')
print(f'  ⚪ Neutras: {analise_vale["noticias_neutras"]}')

## 🤖 Parte 6 — Construção do Agente

Agora vamos montar o sistema Multi-Agent.
Cada agente tem uma responsabilidade específica, seguindo o padrão ReAct.

In [ ]:
# ============================================================
# DIAGRAMA DA ARQUITETURA MULTI-AGENT
# ============================================================

print("""
╔════════════════════════════════════════════════════════╗
║          ARQUITETURA MULTI-AGENT — QUANTUM FINANCE     ║
╠════════════════╦═══════════════╦════════════════════════╣
║  NEWS AGENT    ║ TECHNICAL     ║  DECISION AGENT        ║
║                ║ AGENT         ║                        ║
║ 1. Busca RSS   ║ 1. yfinance   ║ 1. Recebe análises     ║
║ 2. Sentimento  ║ 2. RSI/MACD   ║ 2. Pondera indicadores ║
║ 3. Score       ║ 3. Bollinger  ║ 3. Gera recomendação   ║
║                ║ 4. Volume     ║ 4. Explica raciocínio  ║
╠════════════════╩═══════════════╬════════════════════════╣
║      PERCEPÇÃO + RACIOCÍNIO    ║      AÇÃO              ║
╚════════════════════════════════╩════════════════════════╝
                         ↓
              COMPRAR / VENDER / AGUARDAR
""")

In [ ]:
# Importa os agentes do projeto
from src.agents.news_agent import NewsAgent
from src.agents.technical_agent import TechnicalAgent
from src.agents.decision_agent import DecisionAgent

# Cria o agente de decisão (que orquestra os outros dois)
agente = DecisionAgent()

print('✅ Sistema Multi-Agent inicializado!')
print('   - NewsAgent: pronto')
print('   - TechnicalAgent: pronto')
print('   - DecisionAgent: pronto')

## 💡 Parte 7 — Geração de Recomendações

Agora o agente analisa cada ação e gera as recomendações!
Vamos ver o processo passo a passo para a VALE3.

In [ ]:
# Análise completa da VALE3 passo a passo
print('🔍 ANÁLISE PASSO A PASSO — VALE3')
print('='*50)

# PASSO 1: NewsAgent
print('\n📰 PASSO 1: NewsAgent coletando notícias...')
news_agent = NewsAgent()
resultado_news = news_agent.executar('VALE3')
print(f'   Raciocínio: {resultado_news["raciocinio"]}')
print(f'   Resultado: {resultado_news["resumo"]}')

# PASSO 2: TechnicalAgent
print('\n📊 PASSO 2: TechnicalAgent calculando indicadores...')
tech_agent = TechnicalAgent()
resultado_tecnico = tech_agent.executar('VALE3')
print(f'   Raciocínio: {resultado_tecnico["raciocinio"]}')
print(f'   Resultado: {resultado_tecnico["resumo"]}')
print(f'   Tendência: {resultado_tecnico["tendencia_geral"]}')

In [ ]:
# PASSO 3: DecisionAgent gerando a recomendação
print('\n🧠 PASSO 3: DecisionAgent tomando a decisão...')
from src.recommendation.engine import gerar_recomendacao

recomendacao_vale = gerar_recomendacao(
    'VALE3',
    resultado_tecnico['indicadores'],
    resultado_news['analise_sentimento']
)

# Exibe o resultado em formato JSON
resultado_json = {
    'ticker': recomendacao_vale['ticker'],
    'recommendation': recomendacao_vale['recommendation'],
    'confidence': recomendacao_vale['confidence'],
    'reasoning': recomendacao_vale['reasoning'][:150] + '...'
}

print('\n📋 RESULTADO:')
print(json.dumps(resultado_json, ensure_ascii=False, indent=2))

In [ ]:
# Agora analisa TODOS os tickers de uma vez!
print('🚀 Analisando todos os tickers...')
print('(isso pode levar alguns segundos)\n')

resultados_todos = agente.analisar_todos()

## 📊 Parte 8 — Visualizações com Plotly

Agora vamos criar gráficos interativos pra visualizar os dados e os indicadores.

In [ ]:
# ============================================================
# GRÁFICO 1: Candlestick + Médias Móveis + Bollinger Bands
# ============================================================

def criar_grafico_completo(ticker, df):
    """Cria um gráfico completo com candlestick e indicadores."""
    
    # Calcula os indicadores
    indicadores = calcular_todos_indicadores(df)
    series = indicadores['series']
    
    # Cria subplots: preço, RSI e MACD
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=[f'{ticker} — Preço e Indicadores', 'RSI (14)', 'MACD'],
        row_heights=[0.6, 0.2, 0.2]
    )
    
    # --- CANDLESTICK ---
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['Open'], high=df['High'],
        low=df['Low'], close=df['Close'],
        name='Preço',
        increasing_line_color='#26a69a',
        decreasing_line_color='#ef5350'
    ), row=1, col=1)
    
    # --- MÉDIAS MÓVEIS ---
    fig.add_trace(go.Scatter(
        x=df.index, y=series['sma_20'],
        name='SMA 20', line=dict(color='blue', width=1.5)
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=df.index, y=series['sma_50'],
        name='SMA 50', line=dict(color='orange', width=1.5)
    ), row=1, col=1)
    
    # --- BOLLINGER BANDS ---
    fig.add_trace(go.Scatter(
        x=df.index, y=series['bb_superior'],
        name='BB Superior', line=dict(color='gray', width=1, dash='dash'),
        opacity=0.7
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=df.index, y=series['bb_inferior'],
        name='BB Inferior', line=dict(color='gray', width=1, dash='dash'),
        fill='tonexty', fillcolor='rgba(128,128,128,0.1)',
        opacity=0.7
    ), row=1, col=1)
    
    # --- RSI ---
    fig.add_trace(go.Scatter(
        x=df.index, y=series['rsi'],
        name='RSI', line=dict(color='purple', width=2)
    ), row=2, col=1)
    
    # Linhas de referência do RSI (70 e 30)
    fig.add_hline(y=70, line_dash='dash', line_color='red', opacity=0.5, row=2, col=1)
    fig.add_hline(y=30, line_dash='dash', line_color='green', opacity=0.5, row=2, col=1)
    fig.add_hline(y=50, line_dash='dot', line_color='gray', opacity=0.3, row=2, col=1)
    
    # --- MACD ---
    fig.add_trace(go.Scatter(
        x=df.index, y=series['macd'],
        name='MACD', line=dict(color='blue', width=1.5)
    ), row=3, col=1)
    
    fig.add_trace(go.Scatter(
        x=df.index, y=series['signal'],
        name='Signal', line=dict(color='red', width=1.5)
    ), row=3, col=1)
    
    # Histograma do MACD
    cores_hist = ['green' if v >= 0 else 'red' for v in series['histograma']]
    fig.add_trace(go.Bar(
        x=df.index, y=series['histograma'],
        name='Histograma', marker_color=cores_hist, opacity=0.5
    ), row=3, col=1)
    
    # Layout do gráfico
    fig.update_layout(
        title=f'Análise Técnica — {ticker}',
        height=800,
        showlegend=True,
        xaxis_rangeslider_visible=False,
        template='plotly_white'
    )
    
    return fig

# Importa a função de calcular indicadores
from src.indicators.technical import calcular_todos_indicadores

# Gera o gráfico pra VALE3
fig_vale = criar_grafico_completo('VALE3', dados_mercado['VALE3'])
fig_vale.show()
print('📊 Gráfico da VALE3 exibido!')

In [ ]:
# ============================================================
# GRÁFICO 2: Volume com Média Móvel
# ============================================================

def criar_grafico_volume(ticker, df):
    """Gráfico de volume com média móvel de 20 dias."""
    
    volume = df['Volume']
    volume_medio = volume.rolling(20).mean()
    
    fig = go.Figure()
    
    # Barras de volume coloridas (verde = alta, vermelho = baixa)
    cores = ['green' if df['Close'].iloc[i] >= df['Open'].iloc[i] else 'red'
             for i in range(len(df))]
    
    fig.add_trace(go.Bar(
        x=df.index, y=volume,
        name='Volume', marker_color=cores, opacity=0.7
    ))
    
    fig.add_trace(go.Scatter(
        x=df.index, y=volume_medio,
        name='Volume Médio (20d)',
        line=dict(color='blue', width=2)
    ))
    
    fig.update_layout(
        title=f'Volume de Negociações — {ticker}',
        xaxis_title='Data',
        yaxis_title='Volume',
        template='plotly_white',
        height=400
    )
    
    return fig

fig_volume = criar_grafico_volume('VALE3', dados_mercado['VALE3'])
fig_volume.show()

In [ ]:
# ============================================================
# GRÁFICO 3: Comparação de RSI entre todos os tickers
# ============================================================

fig_rsi = go.Figure()

for ticker, df in dados_mercado.items():
    rsi = calcular_rsi(df['Close'])
    fig_rsi.add_trace(go.Scatter(
        x=df.index, y=rsi, name=ticker, mode='lines'
    ))

# Linhas de referência
fig_rsi.add_hline(y=70, line_dash='dash', line_color='red',
                   annotation_text='Sobrecomprado (70)')
fig_rsi.add_hline(y=30, line_dash='dash', line_color='green',
                   annotation_text='Sobrevendido (30)')

fig_rsi.update_layout(
    title='Comparação de RSI — Todos os Tickers',
    xaxis_title='Data',
    yaxis_title='RSI',
    template='plotly_white',
    height=400
)

fig_rsi.show()

In [ ]:
# ============================================================
# GRÁFICO 4: Dashboard das Recomendações
# ============================================================

if resultados_todos:
    # Prepara os dados para o gráfico
    tickers_rec = [r['ticker'] for r in resultados_todos if 'ticker' in r]
    recomendacoes = [r['recommendation'] for r in resultados_todos if 'recommendation' in r]
    confidencas = [r.get('confidence', 0) * 100 for r in resultados_todos if 'recommendation' in r]
    
    # Cores por recomendação
    cores_rec = {'COMPRAR': '#26a69a', 'VENDER': '#ef5350', 'AGUARDAR': '#ffd54f', 'ERRO': '#bdbdbd'}
    cores = [cores_rec.get(r, '#bdbdbd') for r in recomendacoes]
    
    fig_rec = go.Figure(data=[
        go.Bar(
            x=tickers_rec,
            y=confidencas,
            text=[f'{r}\n{c:.0f}%' for r, c in zip(recomendacoes, confidencas)],
            textposition='inside',
            marker_color=cores,
            textfont=dict(size=14, color='white')
        )
    ])
    
    fig_rec.update_layout(
        title='Dashboard de Recomendações — QuantumFinance',
        xaxis_title='Ativo',
        yaxis_title='Nível de Confiança (%)',
        yaxis_range=[0, 100],
        template='plotly_white',
        height=400
    )
    
    fig_rec.show()
    print('✅ Dashboard de recomendações exibido!')

## 🔄 Parte 9 — Backtest Simples

Vamos testar se a estratégia do agente teria funcionado no passado.
Comparamos com Buy & Hold (comprar e segurar).

In [ ]:
# ============================================================
# BACKTEST SIMPLES
#
# Estratégia simulada:
# - A cada semana, calculamos os indicadores com dados históricos
# - Se o RSI < 35 e MACD bullish → COMPRAR (entramos)
# - Se o RSI > 65 e MACD bearish → VENDER (saímos)
# - Senão → AGUARDAR (ficamos na mesma)
#
# Nota: Este é um backtest MUITO simplificado pra fins didáticos.
# Backtests reais consideram custos de corretagem, slippage, etc.
# ============================================================

def backtest_simples(df, ticker='ATIVO'):
    """Backtest da estratégia baseada em RSI + MACD."""
    
    closes = df['Close'].copy()
    
    # Calcula os indicadores no histórico completo
    rsi_serie = calcular_rsi(closes)
    macd_serie, signal_serie, _ = calcular_macd(closes)
    
    # Simula as operações
    capital = 10000.0     # Capital inicial: R$ 10.000
    acoes = 0.0            # Começa sem ações
    em_posicao = False     # Flag: estamos comprados?
    operacoes = []         # Histórico de operações
    capital_historico = [] # Evolução do capital
    
    preco_compra = 0.0
    acertos = 0
    erros = 0
    
    # Percorre cada dia (a partir do dia 50 pra ter indicadores calculados)
    for i in range(50, len(closes)):
        preco = closes.iloc[i]
        rsi = rsi_serie.iloc[i]
        macd = macd_serie.iloc[i]
        signal = signal_serie.iloc[i]
        
        # Sinal de COMPRA: RSI baixo + MACD bullish + não estamos comprados
        if rsi < 35 and macd > signal and not em_posicao:
            # Compra
            acoes = capital / preco
            preco_compra = preco
            capital = 0
            em_posicao = True
            operacoes.append({'tipo': 'COMPRA', 'preco': preco, 'data': closes.index[i]})
        
        # Sinal de VENDA: RSI alto + MACD bearish + estamos comprados
        elif rsi > 65 and macd < signal and em_posicao:
            # Vende
            capital = acoes * preco
            variacao = (preco - preco_compra) / preco_compra
            if variacao > 0:
                acertos += 1
            else:
                erros += 1
            acoes = 0
            em_posicao = False
            operacoes.append({'tipo': 'VENDA', 'preco': preco, 'data': closes.index[i], 'resultado': f'{variacao:+.2%}'})
        
        # Calcula o valor atual do portfólio
        valor_atual = capital + (acoes * preco)
        capital_historico.append(valor_atual)
    
    # Se ainda estamos comprados, vende no último dia
    if em_posicao:
        capital = acoes * closes.iloc[-1]
        acoes = 0
    
    # Métricas finais
    total_operacoes = acertos + erros
    acuracia = (acertos / total_operacoes * 100) if total_operacoes > 0 else 0
    retorno_agente = ((capital - 10000) / 10000) * 100
    
    # Buy & Hold: simples compra no dia 50 e vende no último
    retorno_buyhold = ((closes.iloc[-1] - closes.iloc[50]) / closes.iloc[50]) * 100
    
    # Sharpe simplificado (retorno / desvio padrão dos retornos diários)
    retornos_diarios = pd.Series(capital_historico).pct_change().dropna()
    sharpe = (retornos_diarios.mean() / retornos_diarios.std()) * (252 ** 0.5) if retornos_diarios.std() > 0 else 0
    
    return {
        'ticker': ticker,
        'capital_inicial': 10000,
        'capital_final': round(capital, 2),
        'retorno_agente': round(retorno_agente, 2),
        'retorno_buyhold': round(retorno_buyhold, 2),
        'total_operacoes': total_operacoes,
        'acertos': acertos,
        'erros': erros,
        'acuracia': round(acuracia, 1),
        'sharpe': round(sharpe, 2),
        'operacoes': operacoes,
        'capital_historico': capital_historico,
        'datas': list(closes.index[50:]),
    }

# Roda o backtest pra todos os tickers
print('⏳ Rodando backtest...')
resultados_backtest = {}

for ticker, df in dados_mercado.items():
    resultado = backtest_simples(df, ticker)
    resultados_backtest[ticker] = resultado
    print(f'  {ticker}: Retorno Agente {resultado["retorno_agente"]:+.1f}% | '
          f'Buy&Hold {resultado["retorno_buyhold"]:+.1f}% | '
          f'Acurácia {resultado["acuracia"]:.0f}% | '
          f'Sharpe {resultado["sharpe"]:.2f}')

print('\n✅ Backtest concluído!')

In [ ]:
# Gráfico comparativo: Agente vs Buy & Hold
fig_backtest = go.Figure()

for ticker, resultado in resultados_backtest.items():
    if resultado['capital_historico']:
        fig_backtest.add_trace(go.Scatter(
            x=resultado['datas'],
            y=resultado['capital_historico'],
            name=f'{ticker} (Agente)',
            mode='lines'
        ))

fig_backtest.add_hline(y=10000, line_dash='dash', line_color='gray',
                        annotation_text='Capital inicial (R$ 10.000)')

fig_backtest.update_layout(
    title='Backtest — Evolução do Capital (Estratégia do Agente)',
    xaxis_title='Data',
    yaxis_title='Capital (R$)',
    template='plotly_white',
    height=450
)

fig_backtest.show()

# Tabela comparativa
print('\n📊 TABELA COMPARATIVA — AGENTE vs BUY & HOLD')
print('='*65)
print(f'  {"TICKER":<8} {"AGENTE":>10} {"BUY&HOLD":>10} {"ACURÁCIA":>10} {"SHARPE":>8}')
print(f'  {"-"*58}')
for ticker, r in resultados_backtest.items():
    print(f'  {ticker:<8} {r["retorno_agente"]:>+9.1f}% {r["retorno_buyhold"]:>+9.1f}% {r["acuracia"]:>9.0f}% {r["sharpe"]:>8.2f}')

## 📝 Parte 10 — Conclusões

### O que aprendemos?

1. **AI Agents** podem ser aplicados ao mercado financeiro combinando múltiplas fontes de dados

2. **Arquitetura Multi-Agent** permite separar responsabilidades:
   - NewsAgent → análise de sentimento
   - TechnicalAgent → indicadores técnicos
   - DecisionAgent → decisão final

3. **Padrão ReAct** (Reason + Act) torna o agente explicável — ele não só decide, mas justifica

4. **Análise de sentimento** com VADER é eficaz para textos financeiros, mesmo em português

5. **Indicadores técnicos** sozinhos não garantem acerto — a combinação com notícias melhora a qualidade

### Limitações do projeto

- Dados gratuitos do Yahoo Finance têm delay (não são real-time)
- VADER não é especializado em português/financeiro (FinBERT seria melhor)
- Backtest simples não considera custos de corretagem ou impostos
- O agente não executa ordens reais (seria necessário integrar com uma corretora)

### Próximos passos

- Integrar FinBERT para análise de sentimento mais precisa
- Adicionar mais fontes de dados (notícias em tempo real, dados macroeconômicos)
- Implementar gestão de risco (stop loss, diversificação)
- Conectar com API de corretora para execução real

In [ ]:
# Relatório final de todas as recomendações
print('='*60)
print('  RELATÓRIO FINAL — QUANTUM FINANCE')
print(f'  Gerado em: {datetime.now().strftime("%d/%m/%Y %H:%M")}')
print('='*60)

for r in resultados_todos:
    if 'recommendation' in r:
        emoji = {'COMPRAR': '🟢', 'VENDER': '🔴', 'AGUARDAR': '🟡'}.get(r['recommendation'], '⚪')
        print(f'\n{emoji} {r["ticker"]}: {r["recommendation"]} (confiança: {r.get("confidence",0):.0%})')
        print(f'   {r.get("reasoning", "")[:120]}...')

print('\n⚠️  AVISO: Este projeto é acadêmico. Não use para decisões financeiras reais!')
print('='*60)